In [1]:
import torch
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.decaf.adapter import KatabaticDECAF # Changed to DECAF Adapter
from utils import discretize_preprocess

# --- Configuration ---
DATASET = "adult"
raw_path = f"raw_data/{DATASET}.csv"
discretized_path = f"discretized_data/{DATASET}.csv"
output_dir = f"sample_data/{DATASET}"
synthetic_dir = f"synthetic/{DATASET}/decaf" # Updated directory
real_test_dir = f"sample_data/{DATASET}"

# Device Config
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# User Input
protected_col = input("Protected Attribute (S) [default 'sex']: ").strip() or "sex"
target_col = input("Target Attribute (Y) [default 'class']: ").strip() or "class"

# --- DECAF Specific: Causal DAG ---
# Define causal edges: [Parent, Child]
# This DAG represents standard assumptions for the Adult dataset.
# DECAF uses this to mask the generator weights.
adult_dag = [
    ['age', 'marital-status'],
    ['sex', 'marital-status'],
    ['sex', 'education'],
    ['race', 'education'],
    ['education', 'occupation'],
    ['education', 'hours-per-week'],
    ['marital-status', 'relationship'],
    ['occupation', 'class'],
    ['hours-per-week', 'class'],
    # Direct edge for protected attribute -> target (to be debiased later)
    [protected_col, target_col] 
]

# --- Model Configuration ---
model_config = {
    # DECAF Hyperparameters
    "epochs": 50,
    "batch_size": 64,
    "dag": adult_dag,  # Pass the causal graph
    
    # Fairness Config (Required for the EVALUATOR, not the model training)
    "fairness_config": {
        "S": protected_col,
        "Y": target_col,
        "S_under": "0",
        "Y_desire": "1"
    }
}

# --- 1. Preprocess ---
print("Discretizing data...")
discretize_preprocess(
    file_path=raw_path,
    output_path=discretized_path,
    bins=10,
    strategy='uniform'
)

# --- 2. Run Pipeline ---
# Instantiate pipeline with the DECAF Adapter class
pipeline = TrainTestSplitPipeline(model=KatabaticDECAF)

print(f"Starting pipeline for DECAF on {DATASET}")

pipeline.run(
    input_csv=discretized_path,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
    **model_config
)

/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Discretizing data...
Preprocessing: raw_data/adult.csv
Saved preprocessed discrete dataset to: discretized_data/adult.csv
Starting pipeline for DECAF on adult
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Loading DECAF training data from: sample_data/adult
Initializing DECAF (Dims:15, DAG Edges:10)...
Training DECAF on 26048 rows for 50 epochs...


Training DECAF:   0%|          | 0/50 [00:00<?, ?it/s]/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Training DECAF: 100%|██████████| 50/50 [05:38<00:00,  6.76s/it, d_loss=-0.449, g_loss=-0.736] 


Generating DECAF synthetic data to: synthetic/adult/decaf
Saved split artifacts: x_synth ((1000, 14)), y_synth ((1000,))


/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/adult/decaf_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7321
F1 Score: 0.6971
AUC: 0.5923

MLP:
Accuracy: 0.7487
F1 Score: 0.6693
AUC: 0.5804

RF:
Accuracy: 0.7566
F1 Score: 0.6563
AUC: 0.4757

XGBoost:
Accuracy: 0.7433
F1 Score: 0.6624
AUC: 0.6790


'Train test split pipeline executed successfully.'